In [3]:
!pip install -q xgboost joblib

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import joblib
from google.colab import files
print("✅ Libraries imported.")

✅ Libraries imported.


In [4]:
print("📥 Loading Ames Housing dataset...")
X, y = fetch_openml(name="ames_housing", version=1, return_X_y=True, as_frame=True)
print(f"✅ Dataset loaded! Shape: {X.shape}")

📥 Loading Ames Housing dataset...
✅ Dataset loaded! Shape: (2930, 80)


In [5]:
selected_features = [
    "Gr_Liv_Area",      # Above-grade living area (sq ft)
    "Year_Built",       # Original construction year
    "Total_Bsmt_SF",    # Total basement area (sq ft)
    "Full_Bath",        # Full bathrooms above grade
    "Half_Bath",        # Half bathrooms above grade
    "Bedroom_AbvGr",    # Bedrooms above grade
    "Garage_Cars",      # Garage capacity
    "Lot_Area",         # Lot size (sq ft)
    "Fireplaces",       # Number of fireplaces
    "Overall_Qual",     # Overall quality (1-10)
    "Overall_Cond"      # Overall condition (1-10)
]

X_selected = X[selected_features].copy()
print(f"✅ Selected {len(selected_features)} features.")
print(f"   Shape: {X_selected.shape}")

✅ Selected 11 features.
   Shape: (2930, 11)


In [6]:
for col in X_selected.columns:
    if X_selected[col].isnull().sum() > 0:
        median_val = X_selected[col].median()
        X_selected[col].fillna(median_val, inplace=True)
        print(f"   Filled missing values in '{col}' with median: {median_val}")

print("✅ Missing values handled.")

✅ Missing values handled.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)
print(f"✅ Split: {len(X_train)} training, {len(X_test)} testing.")

✅ Split: 2344 training, 586 testing.


In [8]:
print("🤖 Training XGBoost Regressor...")

model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

print("✅ Model trained!")


🤖 Training XGBoost Regressor...
✅ Model trained!


In [9]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"📊 RMSE: ${rmse:,.2f}")
print(f"📊 R² Score: {r2:.4f}")

📊 RMSE: $28,951.35
📊 R² Score: 0.8955


In [10]:
importance_df = pd.DataFrame({
    'Feature': selected_features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n📊 Feature Importance:")
print(importance_df)


📊 Feature Importance:
          Feature  Importance
9    Overall_Qual    0.679827
6     Garage_Cars    0.086184
0     Gr_Liv_Area    0.045138
3       Full_Bath    0.038654
2   Total_Bsmt_SF    0.036448
8      Fireplaces    0.031981
4       Half_Bath    0.019183
1      Year_Built    0.018938
5   Bedroom_AbvGr    0.015300
7        Lot_Area    0.015083
10   Overall_Cond    0.013264


In [12]:
joblib.dump(model, 'xgboost_ames.joblib')
print("💾 Saved: xgboost_ames.joblib")

files.download('xgboost_ames.joblib')
print("🎉 Download started!")

💾 Saved: xgboost_ames.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Download started!
